# 📘 Aula 10b: Classificação de Textos com IMDB – Vetorização Binária com Trigramas (3-gram)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula10b_binary_3gram.ipynb)

Neste caderno, realizamos um experimento rigoroso de **Classificação de Sentimentos (Sentiment Analysis)** sobre o clássico dataset **Large Movie Review Dataset (Stanford IMDB)**. O conjunto contém 50.000 resenhas cinematográficas em inglês anotadas polarmente como positivas ou negativas.

O foco específico desta aula é a representação textual via **Trigramas Binários (Trios de Palavras Consecutivas)**.

No modo Trigrama (`ngrams=3`), o vocabulário expande para trios de palavras (ex: `'not very good'`, `'one of the'`). Avaliaremos experimentalmente se aumentar o contexto para 3 palavras traz ganhos reais ou satura o teto de 20.000 tokens máximos.

---

### 🎯 Objetivo da Aula
1. Baixar e estruturar o corpus de resenhas IMDB em disco via `curl` e `tar`.
2. Remover dados não rotulados (`unsup`) e inspecionar resenhas reais em texto bruto.
3. Criar uma partição balanceada de validação (20% do treino) utilizando `pathlib` e semente fixa.
4. Fazer streaming de dados diretamente dos arquivos de texto usando `keras.utils.text_dataset_from_directory`.
5. Configurar a camada `TextVectorization` com `ngrams=3, output_mode='multi_hot'` para um vocabulário de até 20.000 tokens.
6. Treinar uma rede neural densa (`Dense(16) -> Dropout(0.5) -> Dense(1, sigmoid)`) e avaliar a acurácia e perda no conjunto de teste independente.

---

### 💡 A Intenção Pedagógica do Professor
* **Estudo Comparativo Controlado**: O Professor Boldt mantém a arquitetura neural, os hiperparâmetros e as partições de dados rigorosamente idênticas entre as 4 variações da aula 10b, isolando exclusivamente o impacto da técnica de vetorização (Unigrama vs Bigrama vs Trigrama vs TF-IDF) no resultado final.
* **Streaming de Texto Real**: Demonstrar que não é necessário carregar 50.000 arquivos de texto em memória de uma vez só: o TensorFlow lida com streaming em disco de forma otimizada.

---

### 🧭 Trilha da Aula
1. [Download e Extração do Dataset aclImdb](#1-download-e-extração-do-dataset-aclimdb)
2. [Limpeza do Diretório Não Supervisionado](#2-remoção-de-dados-não-supervisionados)
3. [Inspeção de uma Resenha Real Positiva](#3-inspeção-de-uma-resenha-real)
4. [Particionamento da Validação com pathlib](#4-criação-da-partição-de-validação)
5. [Carregamento com text_dataset_from_directory](#5-streaming-com-text_dataset_from_directory)
6. [Configuração da Camada TextVectorization (Trigramas Binários (Trios de Palavras Consecutivas))](#6-vetorização-textual)
7. [Construção do Classificador Neural com Dropout](#7-construção-do-modelo-neural)
8. [Treinamento, Checkpoint e Avaliação no Teste](#8-treinamento-e-avaliação-no-teste)


## 1. As Perguntas Centrais do Processamento de Linguagem Natural

* *"Qual é o tópico deste texto?"* (classificação de tópicos)
* *"Este texto contém conteúdo abusivo?"* (moderação e filtragem)
* *"Este comentário expressa sentimento positivo ou negativo?"* **(Análise de Sentimentos - nosso foco aqui!)**
* *"Qual deveria ser a próxima palavra nesta frase incompleta?"* (modelagem de linguagem / LLMs)
* *"Como você diria isso em alemão?"* (tradução automática)
* *"Como você resumiria este artigo em um parágrafo?"* (sumarização)


## 2. Download e Extração do Dataset aclImdb

**O que estamos fazendo**: Baixar o pacote compactado oficial do Stanford AI Lab contendo o corpus do IMDB.

**Por que fazer assim**: Dataset padrão ouro para benchmarking de análise de sentimentos em texto livre em inglês.



In [ ]:
# Baixa o pacote oficial do corpus de filmes aclImdb
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
# Extrai silenciosamente o arquivo compactado tar.gz
!tar -xf aclImdb_v1.tar.gz


## 3. Remoção de Dados Não Supervisionados

**O que estamos fazendo**: Excluir a pasta `train/unsup` contendo 50.000 resenhas não rotuladas.

**Por que fazer assim**: Como realizaremos aprendizado supervisionado binário (positivo vs negativo), resenhas sem rótulo atrapalhariam o carregador automático de pastas.



In [ ]:
# Remove a subpasta de resenhas não rotuladas
!rm -r aclImdb/train/unsup


## 4. Inspeção de uma Resenha Real

**O que estamos fazendo**: Exibir o conteúdo textual da resenha de treino positiva `4077_10.txt`.

**Por que fazer assim**: Permite examinar a linguagem coloquial, gírias e pontuações do público cinéfilo.



In [ ]:
# Exibe o texto cru de uma resenha positiva do IMDB
!cat aclImdb/train/pos/4077_10.txt


## 5. Criação da Partição de Validação com pathlib

**O que estamos fazendo**: Separar aleatoriamente 20% das resenhas de cada categoria (`pos` e `neg`) para criar a nova pasta `aclImdb/val`.

**Por que fazer assim**: O dataset original só veio com `train` (25.000) e `test` (25.000). Precisamos de validação independente (5.000 resenhas) para monitorar o treinamento e salvar os melhores checkpoints.

> [!NOTE]
> Fixamos a semente `random.Random(1337)` para garantir separação idêntica e reprodutível.



In [ ]:
import os, pathlib, shutil, random

# Define caminhos para a base e pastas de treino e validação
base_dir = pathlib.Path('aclImdb')
val_dir = base_dir / 'val'
train_dir = base_dir / 'train'

# Para cada polaridade ('neg' e 'pos'), move 20% das amostras de treino para validação
for category in ('neg', 'pos'):
  os.makedirs(val_dir / category, exist_ok=True)
  files = os.listdir(train_dir / category)
  # Embaralha com semente fixa
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files)) # 20% das 12.500 resenhas = 2.500 por classe (5.000 total)
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)
print('Partição de validação criada com sucesso!')


## 6. Streaming com text_dataset_from_directory

**O que estamos fazendo**: Instanciar iteradores de texto para `train`, `val` e `test` com lotes de 32 resenhas.

**Por que fazer assim**: O Keras lê os arquivos de texto diretamente das pastas em lotes sem sobrecarregar a memória RAM.



In [ ]:
from tensorflow import keras
batch_size = 32

# Cria datasets iteráveis diretamente a partir da árvore de diretórios em disco
print('Carregando partição de treino...')
train_ds = keras.utils.text_dataset_from_directory(
    'aclImdb/train', batch_size=batch_size
)
print('Carregando partição de validação...')
val_ds = keras.utils.text_dataset_from_directory(
    'aclImdb/val', batch_size=batch_size
)
print('Carregando partição de teste...')
test_ds = keras.utils.text_dataset_from_directory(
    'aclImdb/test', batch_size=batch_size
)


## 7. Inspeção dos Lotes de Texto Bruto

**O que estamos fazendo**: Extrair um lote e imprimir formato e tipo de dados das strings e dos rótulos supervisionados.

**Por que fazer assim**: As entradas são tensores de strings (`tf.string`) e os rótulos são inteiros binários (`0` para negativo, `1` para positivo).



In [ ]:
# Inspeciona a estrutura do lote de texto
for inputs, targets in train_ds:
  print('Formato do lote de textos (inputs.shape):  ', inputs.shape)
  print('Tipo de dados das entradas (inputs.dtype):  ', inputs.dtype)
  print('Formato do lote de rótulos (targets.shape):', targets.shape)
  print('Tipo de dados dos rótulos (targets.dtype): ', targets.dtype)
  print('\nExemplo de texto bruto da primeira amostra:\n', inputs[0].numpy().decode('utf-8')[:300], '...')
  print('Rótulo supervisionado da primeira amostra:  ', targets[0].numpy())
  break


## 8. Configuração da Camada TextVectorization: Trigramas Binários (Trios de Palavras Consecutivas)

**O que estamos fazendo**: Configurar `TextVectorization` com `ngrams=3, output_mode='multi_hot'`, limitar ao vocabulário de 20.000 tokens mais expressivos, ajustar com `.adapt()` no treino e mapear em paralelo (`num_parallel_calls=4`).

**Por que fazer assim**: Transforma os textos variáveis em vetores numéricos de tamanho fixo 20.000 via Trigramas Binários (Trios de Palavras Consecutivas).

> [!TIP]
> O mapeamento assíncrono com `.map(..., num_parallel_calls=4)` processa a vetorização em múltiplos núcleos de CPU!



In [ ]:
from tensorflow.keras.layers import TextVectorization

text_vectorization = TextVectorization(max_tokens=20000,
                                       ngrams=3,
                                       output_mode="multi_hot")

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

binary_3gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_3gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_3gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)


## 9. Inspeção dos Vetores Gerados

**O que estamos fazendo**: Verificar as dimensões do lote após a vetorização.

**Por que fazer assim**: Cada resenha foi convertida com sucesso em um vetor denso/multi-hot de dimensão 20.000.



In [ ]:
# Inspeciona o formato dos dados já vetorizados (binary_3gram)
for inputs, targets in binary_3gram_train_ds:
  print('Formato das entradas vetorizadas:', inputs.shape)
  print('Tipo das entradas vetorizadas:   ', inputs.dtype)
  print('Formato dos rótulos:             ', targets.shape)
  break


## 10. Construção do Classificador Neural

**O que estamos fazendo**: Definir a fábrica `get_model` com entrada de 20.000 atributos, camada densa de 16 neurônios (ReLU), `Dropout(0.5)` e saída sigmoide.

**Por que fazer assim**: Essa arquitetura concisa e rápida é ideal para dados esparsos de alta dimensionalidade em análise de sentimentos.

> [!NOTE]
> O otimizador `RMSprop` com perda `binary_crossentropy` é a combinação clássica para classificação binária de texto.



In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(max_tokens=20000, hidden_dim=16):
  """
  Constrói rede neural densa com dropout para classificação binária de texto.
  """
  inputs = keras.Input(shape=(max_tokens,), dtype='float32')
  x = layers.Dense(hidden_dim, activation='relu')(inputs)
  x = layers.Dropout(0.5)(x)
  outputs = layers.Dense(1, activation='sigmoid')(x)
  model = keras.Model(inputs, outputs)
  model.compile(optimizer='rmsprop',
                loss='binary_crossentropy',
                metrics=['accuracy'])
  return model

model = get_model()
model.summary()


## 11. Treinamento, Checkpoint e Avaliação no Teste

**O que estamos fazendo**: Treinar o modelo por 10 épocas salvando o checkpoint `binary_3gram.keras` com base na melhor perda de validação e avaliar no teste independente.

**Por que fazer assim**: Permite mensurar objetivamente o poder preditivo desta técnica de representação textual no corpus de teste.



In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint('binary_3gram.keras', save_best_only=True)
]

# Treinamento da rede nas resenhas vetorizadas
model.fit(binary_3gram_train_ds,
          validation_data=binary_3gram_val_ds,
          epochs=10,
          callbacks=callbacks)

# Carrega a melhor versão salva e avalia no teste independente
model = keras.models.load_model('binary_3gram.keras')
test_loss, test_acc = model.evaluate(binary_3gram_test_ds)
print(f'Perda no Teste:    {test_loss:.3f}')
print(f'Acurácia no Teste: {test_acc * 100:.2f}%')


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, aplicamos a representação por **Trigramas Binários (Trios de Palavras Consecutivas)** para Análise de Sentimentos no dataset IMDB:

1. **Estruturação de Textos em Disco**: Carregamos e particionamos 50.000 resenhas brutas do IMDB com `text_dataset_from_directory`.
2. **Peculiaridade de Trigramas Binários (Trios de Palavras Consecutivas)**: Compreendemos como a camada `TextVectorization` transforma o vocabulário em tensores de dimensão 20.000.
3. **Regularização**: O uso de `Dropout(0.5)` evitou a memorização rápida do vocabulário esparso.
4. **Performance Objetiva**: Registramos o desempenho de acurácia no teste que será comparado diretamente com as demais variações (Unigramas, Bigramas, Trigramas e TF-IDF).

---

## 👉 Próximo Passo
No **Lote 10 (Aulas 10f e 10g)**, abandonaremos o modelo Bag-of-Words estático para explorar **Redes Recorrentes LSTM** e **Embeddings Contínuos Pré-treinados (GloVe)**!
